# Un solo spazio per le immagini e le parole

Il codice del capitolo [«Un solo spazio per le immagini e le parole»](https://book.paithon.it/main/VisioneLinguaggio/allineare-due-spazi.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torchvision

## Un solo spazio per le immagini e le parole

[Leggi la pagina](https://book.paithon.it/main/VisioneLinguaggio/allineare-due-spazi.html)


### La loss in dieci righe


In [ ]:
import torchfrom torch import nnimport torch.nn.functional as F# la temperatura si impara: il parametro e' log(1/tau), cosi' la scala,# che si ottiene esponenziando, e' positiva per costruzionelogit_scale = nn.Parameter(torch.tensor(1 / 0.07).log())def loss_contrastiva(emb_img, emb_txt, logit_scale):    """emb_img, emb_txt: due tensori (N, d), una riga per elemento del batch."""    # 1. sulla sfera unitaria: il prodotto scalare diventa un coseno    I = F.normalize(emb_img, dim=-1)    T = F.normalize(emb_txt, dim=-1)    # 2. matrice N x N dei coseni, riscalata dalla temperatura (con il tetto)    scala = logit_scale.exp().clamp(max=100.0)    logits = scala * (I @ T.t())    # 3. la risposta giusta e' sempre sulla diagonale: 0, 1, 2, ... N-1    bersagli = torch.arange(len(I), device=I.device)    # 4. una cross-entropy sulle righe, una sulle colonne, e si media    perdita_i2t = F.cross_entropy(logits, bersagli)    perdita_t2i = F.cross_entropy(logits.t(), bersagli)    return (perdita_i2t + perdita_t2i) / 2

## Il pezzo più piccolo che si può addestrare

[Leggi la pagina](https://book.paithon.it/main/VisioneLinguaggio/innestare-gli-occhi.html)


### Il connettore in dieci righe


In [ ]:
import torchfrom torch import nnclass Proiettore(nn.Module):    """Porta le feature dell'encoder visivo nello spazio dei token del testo."""    def __init__(self, d_visione: int, d_testo: int):        super().__init__()        self.rete = nn.Sequential(            nn.Linear(d_visione, d_testo),            nn.GELU(),            nn.Linear(d_testo, d_testo),        )    def forward(self, patch: torch.Tensor) -> torch.Tensor:        # patch: (B, N, d_visione) -> (B, N, d_testo). Una patch, un token.        return self.rete(patch)proiettore = Proiettore(d_visione=1024, d_testo=4096)print(sum(p.numel() for p in proiettore.parameters()))  # 20979712

*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

# encoder e llm sono pre-addestrati e congelati; si addestra solo il proiettore
for modulo in (encoder, llm):
    for p in modulo.parameters():
        p.requires_grad = False

with torch.no_grad():
    patch = encoder(immagine)          # (B, 576, 1024): la griglia di feature

token_visivi = proiettore(patch)       # (B, 576, 4096): ora sono "parole"

# llm è un modello causale della libreria transformers: espone la tabella
# degli embedding e accetta gli embedding già calcolati al posto degli id
tabella = llm.get_input_embeddings()
prefisso = tabella(id_prima)           # (B, T1, 4096) il testo che precede
suffisso = tabella(id_dopo)            # (B, T2, 4096) la domanda vera e propria

# la sequenza che entra nel decoder: testo, immagine, testo. Tutto insieme.
ingresso = torch.cat([prefisso, token_visivi, suffisso], dim=1)

uscita = llm(inputs_embeds=ingresso, attention_mask=maschera, labels=etichette)
```


## Un vocabolario solo: fusione tardiva e fusione precoce

[Leggi la pagina](https://book.paithon.it/main/VisioneLinguaggio/fusione-precoce-tardiva.html)


### La terza via: due obiettivi, un modello solo


In [ ]:
import numpy as npdef maschera_mista(segmenti):    """Maschera di attenzione per una sequenza mista di testo e immagini.    segmenti: lista di coppie (tipo, lunghezza), con tipo "testo" o "immagine".    Restituisce una matrice booleana: consentito[i, j] e' True se la posizione    i puo' guardare la posizione j."""    tipi, blocchi = [], []    for indice, (tipo, lunghezza) in enumerate(segmenti):        tipi.extend([tipo] * lunghezza)        # tipo di ogni posizione        blocchi.extend([indice] * lunghezza)   # a quale segmento appartiene    n = len(tipi)    blocchi = np.array(blocchi)    e_immagine = np.array([t == "immagine" for t in tipi])    consentito = np.tril(np.ones((n, n), dtype=bool))     # causale: j <= i    stesso_blocco = blocchi[:, None] == blocchi[None, :]    dentro_immagine = stesso_blocco & e_immagine[:, None] & e_immagine[None, :]    return consentito | dentro_immagine    # nel blocco immagine, anche avantim = maschera_mista([("testo", 3), ("immagine", 4), ("testo", 2)])for riga in m:    print("".join("X" if v else "." for v in riga))

## Il costo del dettaglio

[Leggi la pagina](https://book.paithon.it/main/VisioneLinguaggio/risoluzione-e-dettaglio.html)


### Il conto in venti righe


In [ ]:
import numpy as npPATCH = 14  # lato della patch dell'encoder, in pixeldef token(lato, patch=PATCH):    """Token di un ViT su un'immagine quadrata: una patch, un token."""    return (lato // patch) ** 2def a_tessere(lato, tessera=448, riduzione=1):    """Tiling: riquadri alla risoluzione nativa piu' una miniatura dell'intera    immagine. Restituisce (pezzi, token totali, coppie viste dall'encoder)."""    pezzi = (lato // tessera) ** 2 + 1                 # +1: la miniatura    per_pezzo = token(tessera) // riduzione    return pezzi, pezzi * per_pezzo, pezzi * per_pezzo ** 2lati = np.array([224, 448, 896])n = np.array([token(l) for l in lati])print(f"{'immagine':>13} {'token':>7} {'x token':>9} {'x attenzione':>13}")for lato, t in zip(lati, n):    print(f"{lato:>5} x {lato:<5} {t:>7} {t / n[0]:>8.0f}x {(t / n[0]) ** 2:>12.0f}x")lato = 896pezzi, tot, coppie = a_tessere(lato)_, tot_ps, _ = a_tessere(lato, riduzione=4)   # riduzione=4: pixel shuffle 2x2print(f"\n{lato} x {lato} in tessere da 448:")print(f"  monolitica    {token(lato):>5} token   {token(lato) ** 2:>9} coppie nell'encoder")print(f"  a tessere     {tot:>5} token   {coppie:>9} coppie  ({pezzi} pezzi)")print(f"  l'encoder costa {token(lato) ** 2 / coppie:.1f} volte meno")print(f"  con pixel shuffle al modello di linguaggio arrivano {tot_ps} token")

## La forchetta che non c'era

[Leggi la pagina](https://book.paithon.it/main/VisioneLinguaggio/vedere-quel-che-non-ce.html)


### Una domanda con due sole risposte


In [ ]:
import numpy as np# Un test bilanciato: 1500 domande su oggetti presenti, 1500 su oggetti assenti.verita = np.array([1] * 1500 + [0] * 1500)      # 1 = l'oggetto c'e' davverodef pagella(risposte):    vp = ((risposte == 1) & (verita == 1)).sum()   # dice si', e c'e'    fp = ((risposte == 1) & (verita == 0)).sum()   # dice si', e non c'e'    fn = ((risposte == 0) & (verita == 1)).sum()   # dice no, e invece c'e'    precisione = vp / (vp + fp)    richiamo = vp / (vp + fn)    f1 = 2 * precisione * richiamo / (precisione + richiamo)    return round(float(f1), 3), round(float((risposte == 1).mean()), 3)def guarda_davvero(a):    """Modello che risponde correttamente a una frazione a di ciascuna classe."""    giuste = round(1500 * a)    return np.concatenate([        np.array([1] * giuste + [0] * (1500 - giuste)),      # sui presenti        np.array([0] * giuste + [1] * (1500 - giuste)),      # sugli assenti    ])print(pagella(np.ones(3000, dtype=int)))   # (0.667, 1.0)  dice sempre "si'"print(pagella(guarda_davvero(0.60)))       # (0.6, 0.5)    guarda, e sbaglia moltoprint(pagella(guarda_davvero(0.90)))       # (0.9, 0.5)    guarda bene

### Dalla percezione all'azione


In [ ]:
import numpy as npnp.set_printoptions(precision=4, suppress=True)# Sette gradi di liberta': 3 di traslazione (metri), 3 di rotazione (radianti),# 1 per l'apertura della pinza. Gli estremi sono i quantili all'1% e al 99%# delle dimostrazioni, non il minimo e il massimo.basso = np.array([-0.05, -0.05, -0.05, -0.20, -0.20, -0.20, 0.0])alto  = np.array([ 0.05,  0.05,  0.05,  0.20,  0.20,  0.20, 1.0])N_BIN = 256           # i gradini del righelloPRIMO = 32000 - 256   # gli ultimi 256 identificativi di un vocabolario da 32.000def in_token(a):    """Da un'azione continua a sette identificativi di token."""    frazione = (np.clip(a, basso, alto) - basso) / (alto - basso)   # in [0, 1]    return PRIMO + np.minimum((frazione * N_BIN).astype(int), N_BIN - 1)def in_azione(token):    """E ritorno: il centro del gradino, l'unica cosa che il braccio esegue."""    frazione = (token - PRIMO + 0.5) / N_BIN    return basso + frazione * (alto - basso)a = np.array([0.012, -0.004, 0.021, 0.05, -0.11, 0.0, 1.0])print(in_token(a))                    # [31902 31861 31925 31904 31801 31872 31999]print(in_azione(in_token(a)))         # [ 0.0119 -0.0041  0.0209  0.0508 -0.1102  0.0008  0.998 ]print((alto - basso) / (2 * N_BIN))   # [0.0002 0.0002 0.0002 0.0008 0.0008 0.0008 0.002 ]